# Chapter 37
## Thresholding in PING
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter37.ipynb)

## About this chapter

PING can have sharp boundaries between suppression and participation: a
cell either gets recruited into a gamma cycle or it does not, and the
boundary between the two outcomes can be very narrow. The examples below
isolate a non-reset threshold mechanism, run a baseline PING network to
locate that boundary, magnify the transition where a small timing or drive
change flips a cell's participation, and finally compute the boundary
directly for a single cell driven by a periodic inhibitory conductance.

Thresholding is a network property, not a single-cell property in
isolation: excitation must arrive during the interval left open by
recurrent inhibition. A non-reset reference cell (`NO_RESET`) separates
continuous voltage crossing from an imposed reset, so a "threshold event"
can be told apart from reset dynamics. Near a boundary, a single cell can
miss a cycle, or recruit the I population into the next one, from what
looks like a negligible change in drive or timing.

In the last example, a single RTM-style cell is driven by a fixed periodic
inhibitory conductance trace $g(t)=e^{\cos^4(\pi t/25)}-1$ (scaled to a
target mean $\bar g$) and swept in external drive $I$. The onset drive
$I_L$ (first spike) and the drive $I_R$ at which the cell locks onto the
full 39 Hz rhythm bracket a "thresholding window" $w=I_R-I_L$ that shrinks
as $\bar g$ grows -- stronger inhibition sharpens the threshold.

See [`chapter37.md`](chapter37.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

## NO_RESET: a threshold event without a voltage reset

A single passive membrane is driven by a constant current $I$ while a brief
window of extra "inhibitory" leak (time constant `tau_m_hat`, active for
`epsilon` ms out of every period `T`) repeats periodically. Unlike a
spiking neuron, the voltage is never reset here -- the point is to show
that inhibition speeding up the membrane's effective time constant is, by
itself, enough to carve out a repeating window, with no reset mechanism
involved.

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit


def no_reset_time_constants(tau_m=10., g_bar=1. / 7, T=25., epsilon=10.):
    """Effective membrane time constant during the brief inhibited window
    (tau_m_hat) versus the free membrane time constant (tau_m). Inhibition
    speeds up the membrane, so tau_m_hat < tau_m."""
    tau_m_hat = 1. / (1. / tau_m + g_bar * T / epsilon)
    return tau_m, tau_m_hat


def simulate_no_reset(tau_m=10., I=0.12, T=25., epsilon=10., g_bar=1. / 7,
                       n_cycles=5, v0=1.0):
    """Piecewise-exponential voltage trace: tau_m_hat during the first
    `epsilon` ms of every period T (inhibited), tau_m for the remainder
    (free), repeated n_cycles times. No reset is applied at any point."""
    tau_m, tau_m_hat = no_reset_time_constants(tau_m, g_bar, T, epsilon)
    t_all, v_all = [], []
    for ijk in range(n_cycles):
        t = np.arange(101) / 100 * epsilon
        v = v0 * np.exp(-t / tau_m_hat) + tau_m_hat * I * (1 - np.exp(-t / tau_m_hat))
        t_all.append(t + ijk * T)
        v_all.append(v)
        v0 = v[-1]

        t = np.arange(101) / 100 * (T - epsilon)
        v = v0 * np.exp(-t / tau_m) + tau_m * I * (1 - np.exp(-t / tau_m))
        t_all.append(t + ijk * T + epsilon)
        v_all.append(v)
        v0 = v[-1]

    return np.concatenate(t_all), np.concatenate(v_all), tau_m, tau_m_hat


def plot_no_reset(t, v, T=25., n_cycles=5):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(t, v, '-k', linewidth=2)
    ax.set_xlabel('$t$')
    ax.set_ylabel('$v$')
    ax.axis([0, n_cycles * T, 0, 1])
    plt.tight_layout()
    return fig

In [ ]:
t, v, tau_m, tau_m_hat = simulate_no_reset()
print(f"tau_m = {tau_m:.3f}, tau_m_hat = {tau_m_hat:.3f}")
plot_no_reset(t, v)
plt.show()

In [ ]:
interact(lambda g_bar=1. / 7: plot_no_reset(*simulate_no_reset(g_bar=g_bar)[:2]),
         g_bar=(0.02, 0.5, 0.01));

## PING network (shared by PING_THR_1 and PING_THR_1_ZOOM)

Both examples below use the same RTM-E / WB-I PING network as Chapter 30,
with 200 E cells whose drive `i_ext_e` ramps linearly from 1.2 to 2.0
(instead of being homogeneous or randomly heterogeneous) and all-to-all
E-to-I / I-to-E / I-to-I connectivity. The linear ramp turns the E-cell
index into a drive axis, so the raster directly shows the drive at which
cells stop participating in every cycle -- the thresholding boundary this
chapter is about. The per-timestep update is `@njit`-compiled since it
integrates 250 cells over up to 50000 steps.

In [ ]:
from numpy import exp, tanh
from numba.typed import List


# ------------------------------------------------------------- E cell (RTM)

def m_e_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


def h_e_inf(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


def tau_h_e(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


def n_e_inf(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


def tau_n_e(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


# ------------------------------------------------------------- I cell (WB)

def m_i_inf(v):
    alpha_m = 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))
    beta_m = 4. * exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


def h_i_inf(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


def n_i_inf(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


# --------------------------------------------------------- double-exp synapse

def tau_peak_function(tau_d, tau_r, tau_d_q):
    dt_ = 0.01
    dt05_ = dt_ / 2
    s, t = 0., 0.
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05_ * s_inc
        s_inc_tmp = exp(-(t + dt05_) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt_ * s_inc_tmp
        t = t + dt_
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    tau_d_q_left = 1.
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2


# ------------------------------------------------------- population splay init

def rtm_init_population(i_ext, phi_vec):
    """vectorized rtm_init over a population: each of len(i_ext) RTM
    neurons is integrated (Heun/midpoint) independently until its 3rd
    spike, then (v,h,n) is interpolated at phase phi_vec[i] between the
    2nd and 3rd spikes. Faithfully reproduces the matlab source's bug:
    m_tmp is computed from the pre-half-step v, not from v_tmp."""
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=int)
    done = np.zeros(num, dtype=bool)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = (g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext) / c
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of matlab's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                  + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def make_all_to_all_connectivity(num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii, rng):
    """all-to-all connectivity (p_XY=1 everywhere); weights normalized by
    population size so the total conductance onto a cell is g_hat_XY. RNG
    draws (even when a weight ends up all-zero) match the original
    scripts' draw order so the same seed reproduces the same network."""
    u_ee = rng.random((num_e, num_e))
    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ee = g_hat_ee * (u_ee < 1.0) / num_e if g_hat_ee > 0 else np.zeros((num_e, num_e))
    g_ei = g_hat_ei * (u_ei < 1.0) / num_e
    g_ie = g_hat_ie * (u_ie < 1.0) / num_i
    g_ii = g_hat_ii * (u_ii < 1.0) / num_i
    return g_ee, g_ei, g_ie, g_ii


# ------------------------------------------------------- numba network stepper

@njit
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


@njit
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m = 4. * math.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5.


@njit
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5.


@njit
def _ping_thr_step_loop(m_steps, dt, dt05, num_e, num_i,
                         v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                         tau_r_i, tau_d_i, tau_dq_i,
                         i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
                         v_e, h_e, n_e, m_e, q_e, s_e,
                         v_i, h_i, n_i, m_i, q_i, s_i):
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e)
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    lfp = np.empty(m_steps + 1)
    lfp[0] = v_e.mean()

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_i[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(n_e[j], 4.0) * (-100 - v)
                      + 100 * math.pow(m_e[j], 3.0) * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(n_i[j], 4.0) * (-90 - v)
                      + 35 * math.pow(m_i[j], 3.0) * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * si_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * math.pow(ne_m[j], 4.0) * (-100 - v)
                      + 100 * math.pow(me_m[j], 3.0) * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * math.pow(ni_m[j], 4.0) * (-90 - v)
                      + 35 * math.pow(mi_m[j], 3.0) * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))

        lfp[k] = v_e.mean()

    return e_times, e_indices, i_times, i_indices, lfp


def simulate_ping_thr(t_final=200., seed=63806, g_hat_ee=0., g_hat_ei=0.5,
                       g_hat_ie=1.5, g_hat_ii=0.25, dt=0.01):
    """Shared PING network stepper for PING_THR_1 and PING_THR_1_ZOOM: 200
    RTM E-cells with a linearly ramping drive (1.2-2.0), 50 WB I-cells at a
    constant drive, all-to-all connectivity, and constant (non-splay)
    I-cell initialization."""
    num_e, num_i = 200, 50
    i_ext_e = 1.2 + np.arange(1, num_e + 1) / num_e * 0.8
    sigma_i = 0.00
    rng = np.random.default_rng(seed)
    i_ext_i = 0.5 * np.ones(num_i) * (1 + sigma_i * rng.standard_normal(num_i))

    v_rev_e, v_rev_i = 0., -75.
    tau_r_e, tau_peak_e, tau_d_e = 0.5, 0.5, 3.
    tau_r_i, tau_peak_i, tau_d_i = 0.5, 0.5, 9.
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    g_ee, g_ei, g_ie, g_ii = make_all_to_all_connectivity(
        num_e, num_i, g_hat_ee, g_hat_ei, g_hat_ie, g_hat_ii, rng)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0], iv[:, 1], iv[:, 2]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    t_e, i_e, t_i, i_i, lfp = _ping_thr_step_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i,
        i_ext_e, i_ext_i, g_ee, g_ei, g_ie, g_ii,
        v_e, h_e, n_e, m_e, q_e, s_e,
        v_i, h_i, n_i, m_i, q_i, s_i,
    )
    t_e = np.array(t_e) if len(t_e) else np.empty(0)
    i_e = np.array(i_e, dtype=int) if len(i_e) else np.empty(0, dtype=int)
    t_i = np.array(t_i) if len(t_i) else np.empty(0)
    i_i = np.array(i_i, dtype=int) if len(i_i) else np.empty(0, dtype=int)
    return t_e, i_e, t_i, i_i, np.array(lfp), i_ext_e, num_e, num_i


def plot_ping_thr_raster(t_e, i_e, t_i, i_i, num_e, num_i, t_final, title=""):
    fig, ax = plt.subplots(figsize=(8, 5))
    if len(t_i) > 0:
        ax.plot(t_i, i_i, '.b', markersize=2)
    if len(t_e) > 0:
        ax.plot(t_e, i_e + num_i, '.r', markersize=2)
    ax.plot([0, t_final], [num_i + 0.5, num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([num_i, num_e + num_i])
    ax.axis([0, t_final, 0, num_e + num_i + 1])
    ax.set_xlabel('$t$ [ms]')
    if title:
        ax.set_title(title)
    plt.tight_layout()
    return fig

## PING_THR_1: baseline PING thresholding

`simulate_ping_thr_1` runs the shared network for 200 ms. Because
`i_ext_e` ramps linearly across the 200 E cells, the raster's vertical axis
doubles as a drive axis: the boundary E-cell index above which cells stop
firing every cycle is the thresholding boundary. Try the slider below to
see how strengthening I-to-E inhibition (`g_hat_ie`) raises that
boundary.

In [ ]:
def simulate_ping_thr_1(t_final=200.0, seed=63806, g_hat_ie=1.5):
    return simulate_ping_thr(t_final=t_final, seed=seed, g_hat_ie=g_hat_ie)


t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, lfp, i_ext_e, num_e, num_i = simulate_ping_thr_1()
plot_ping_thr_raster(t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, num_e, num_i, t_final=200.0,
                      title="PING_THR_1")
plt.show()

In [ ]:
interact(lambda g_hat_ie=1.5: plot_ping_thr_raster(
    *simulate_ping_thr_1(g_hat_ie=g_hat_ie)[:4], num_e=200, num_i=50, t_final=200.0,
    title=f"g_hat_ie = {g_hat_ie:.2f}"),
    g_hat_ie=(0.5, 3.0, 0.1));

## PING_THR_1_ZOOM: magnifying the boundary

`simulate_ping_thr_1_zoom` is the same network run out to 500 ms (giving
the boundary cells more cycles to settle into a repeatable pattern); the
plot then zooms into E-cells 72-78 (1-indexed) to inspect, cycle by cycle,
whether each one keeps firing, drops out, or fires irregularly right at the
suppression/participation transition.

In [ ]:
def simulate_ping_thr_1_zoom(t_final=500.0, seed=63806, g_hat_ie=1.5):
    return simulate_ping_thr(t_final=t_final, seed=seed, g_hat_ie=g_hat_ie)


def plot_ping_thr_zoom(t_e, i_e, t_final=500.0, lo=71, hi=77, title=""):
    fig, ax = plt.subplots(figsize=(8, 5))
    mask = (i_e >= lo) & (i_e <= hi)
    ax.plot(t_e[mask], i_e[mask] + 1, '.r', markersize=12)
    ax.axis([50, t_final, lo + 0.5, hi + 1.5])
    ax.set_xlabel('$t$ [ms]')
    ax.set_ylabel('E-cell number')
    if title:
        ax.set_title(title)
    plt.tight_layout()
    return fig

In [ ]:
res_zoom = simulate_ping_thr_1_zoom()
t_e_zoom, i_e_zoom = res_zoom[0], res_zoom[1]
plot_ping_thr_zoom(t_e_zoom, i_e_zoom, t_final=500.0, title="PING_THR_1_ZOOM")
plt.show()

## THRESHOLDING: the threshold construction directly

A single RTM cell is driven by a fixed external current $I$ together with a
periodic inhibitory conductance trace $g(t)=e^{\cos^4(\pi t/25)}-1$ scaled
to a target mean $\bar g$. `bisect_threshold` finds, by bisection, the
drive $I_L$ at which the cell first starts spiking at all, and the higher
drive $I_R$ at which it locks onto the network's full 39 Hz rhythm.
`threshold_width_sweep` repeats this for five values of $\bar g$ and
returns the width $w=I_R-I_L$ of the transition window for each -- w
shrinks roughly geometrically as $\bar g$ grows, i.e. stronger inhibition
sharpens the threshold. The per-drive simulation (5000 ms at `dt=0.01`) is
`@njit`-compiled since dozens of bisection evaluations are needed per
`g_bar`.

In [ ]:
def g(t):
    return np.exp(np.cos(np.pi * t / 25) ** 4) - 1


@njit(fastmath=True)
def m_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit(fastmath=True)
def alpha_h(v):
    return 0.128 * math.exp(-(v + 50) / 18)


@njit(fastmath=True)
def beta_h(v):
    return 4. / (1 + math.exp(-(v + 27) / 5))


@njit(fastmath=True)
def alpha_n(v):
    return 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))


@njit(fastmath=True)
def beta_n(v):
    return 0.5 * math.exp(-(v + 57) / 40)


@njit(fastmath=True)
def _firing_rate_loop(i_ext, g_store, m_steps, dt, dt05, c, g_na, g_k, g_l,
                       v_na, v_k, v_l, v_rev, stop_at_first_spike):
    v, m, h, n = -70., m_inf(-70.), 0.7, 0.6
    num_spikes = 0
    for k in range(m_steps):
        v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 4.0) * (v_k - v)
                  + g_l * (v_l - v) + g_store[k] * (v_rev - v) + i_ext) / c
        h_inc = alpha_h(v) * (1 - h) - beta_h(v) * h
        n_inc = alpha_n(v) * (1 - n) - beta_n(v) * n

        v_tmp = v + dt05 * v_inc
        m_tmp = m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc

        v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp)
                  + g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp)
                  + g_l * (v_l - v_tmp) + (g_store[k] + g_store[k + 1]) * 0.5 * (v_rev - v_tmp) + i_ext) / c
        h_inc = alpha_h(v_tmp) * (1 - h_tmp) - beta_h(v_tmp) * h_tmp
        n_inc = alpha_n(v_tmp) * (1 - n_tmp) - beta_n(v_tmp) * n_tmp

        v_new = v + dt * v_inc
        m = m_inf(v_new)
        h = h + dt * h_inc
        n = n + dt * n_inc

        if v_new < -20 and v >= -20:
            num_spikes += 1
            if stop_at_first_spike:
                break
        v = v_new

    return num_spikes


def firing_rate(i_ext, g_store, stop_at_first_spike=False, t_final=5000., dt=0.01):
    """run t_final ms of the RTM neuron under drive i_ext and the
    precomputed inhibitory conductance trace g_store, and return the
    resulting firing rate. If stop_at_first_spike is set, the simulation
    is cut short as soon as one spike occurs -- valid only when the
    caller merely needs to know that the (monotonically non-decreasing)
    final rate is positive, e.g. the target=1e-9 "any spiking" search
    below, since it never changes that classification."""
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    c, g_k, g_na, g_l = 1., 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.
    v_rev = -75.
    num_spikes = _firing_rate_loop(i_ext, g_store, m_steps, dt, dt05, c,
                                    g_na, g_k, g_l, v_na, v_k, v_l, v_rev,
                                    stop_at_first_spike)
    return num_spikes / t_final * 1000


def bisect_threshold(g_store, target, stop_at_first_spike=False, epsilon=1e-5,
                      t_final=5000., dt=0.01):
    """bisection search for the drive at which the firing rate first
    reaches (>=) target, i.e. the boundary between f==0 (or f<target)
    and f>=target."""
    i_ext_left, i_ext_right = 0., 3.
    while (i_ext_right - i_ext_left) / ((i_ext_right + i_ext_left) / 2) > epsilon:
        i_ext = (i_ext_left + i_ext_right) / 2
        f = firing_rate(i_ext, g_store, stop_at_first_spike, t_final, dt)
        if f < target:
            i_ext_left = i_ext
        else:
            i_ext_right = i_ext
    return (i_ext_left + i_ext_right) / 2


def threshold_width_sweep(g_bar_vec=None, t_final=5000., dt=0.01):
    """for each g_bar in g_bar_vec, find I_L (onset of any firing) and I_R
    (onset of full-rate 39 Hz locked firing) for the periodic inhibitory
    conductance g(t) scaled to mean g_bar, and return the resulting
    thresholding window w = I_R - I_L."""
    if g_bar_vec is None:
        g_bar_vec = 0.15 + np.arange(5) * 0.05
    m_steps = round(t_final / dt)
    t = np.arange(m_steps + 1) * dt
    w = np.zeros(len(g_bar_vec))

    for ijk, g_bar in enumerate(g_bar_vec):
        g_mean = np.mean(g(np.arange(1, 10001) / 10000 * 25))
        g_factor = g_bar / g_mean
        g_store = g(t) * g_factor

        I_L = bisect_threshold(g_store, target=1e-9, stop_at_first_spike=True, t_final=t_final, dt=dt)
        I_R = bisect_threshold(g_store, target=39., t_final=t_final, dt=dt)
        w[ijk] = I_R - I_L

    return g_bar_vec, w, t

In [ ]:
g_bar_vec, w, t = threshold_width_sweep()
ratios = w[:4] / w[1:]
for g_bar_ijk, w_ijk in zip(g_bar_vec, w):
    print(f"g_bar={g_bar_ijk:.2f}  w={w_ijk:.6f}")
print("w =", w)
print("ratios =", ratios)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(g_bar_vec, w, 'o-k')
ax.set_xlabel(r'$\bar g$')
ax.set_ylabel('$w = I_R - I_L$')
plt.tight_layout()
plt.show()